<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
To install RapidFire AI on your own machine, see the <a href="https://oss-docs.rapidfire.ai/en/latest/walkthrough.html">Install and Get Started</a> guide in our docs.
</div>

## RapidFire AI Multimodal RAG Tutorial — `Unstructured` + `Pinecone`

End-to-end multimodal RAG over real-world PDFs, evaluated with RapidFire AI. We:

1. **Parse PDFs** into text / table / image elements with [Unstructured](https://docs.unstructured.io/).
2. **Summarize** each element with an LLM so every modality lands as embeddable text.
3. **Offload** the originals to cloud object storage (GCS here; S3 also supported) so vector-store metadata stays small.
4. **Embed and index** the summaries in a **Pinecone** serverless index.
5. **Retrieve + rerank** top chunks at query time and answer with a vision-capable LLM.
6. Wrap it all in a `RFLangChainRagSpec` so RapidFire can run, track, and (in follow-up tutorials) hyperparallelize it.

Dataset: [**MMDocIR**](https://huggingface.co/datasets/MMDocIR/MMDocIR_Evaluation_Dataset) — 313 PDFs with 1,658 ground-truth Q&A pairs across academic papers, financial reports, government filings, and more.

---

### System dependencies

`hi_res` PDF parsing needs a few native libraries plus two NLTK corpora:

```bash
sudo apt-get install -y libmagic1 poppler-utils tesseract-ocr libxml2 libxslt1-dev
pip install -Uq "unstructured[all-docs]"
pip install nltk
```

```python
import nltk
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")
```

In [ ]:
import getpass

OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API key: ")
GOOGLE_API_KEY = getpass.getpass("Enter your Google API key: ")
ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key: ")
PINECONE_API_KEY = getpass.getpass("Enter your Pinecone API key: ")

In [ ]:
import warnings

# Unstructured + LangChain emit a lot of forward-compat warnings that
# clutter notebook output. Silence them for the duration of the tutorial.
warnings.filterwarnings("ignore")

from rapidfireai import Experiment
from rapidfireai.automl import (
    List,                  # multi-value knob -- "try every value in this list"
    RFLangChainRagSpec,    # declarative spec for a LangChain RAG pipeline
    RFPromptManager,       # manage prompts for a LangChain RAG pipeline
    RFAPIModelConfig,      # remote-LLM (OpenAI/Anthropic/Google/...) config
    RFGridSearch,          # cross-product of every multi-value knob
)
from datasets import Dataset
from typing import List as listtype, Dict, Any

## 1. Load the MMDocIR dataset

We use two artifacts from the MMDocIR repo:

| File | Contents | Used? |
|------|----------|-------|
| `doc_pdfs.rar` | 313 source PDFs (~1 GB) | ✅ indexed and queried |
| `MMDocIR_annotations.jsonl` | Per-document Q&A with ground-truth page IDs | ✅ retrieval eval |
| `MMDocIR_pages.parquet` | Pre-rendered page screenshots (20k rows) | ⏭ re-extract from PDFs |
| `MMDocIR_layouts.parquet` | Pre-detected layout elements (170k rows) | ⏭ Unstructured re-does |

Downloaded via `wget` + `unrar` (install via `apt-get install -y wget unrar` on Debian/Ubuntu, or `brew install wget unar` on macOS — swap `unrar` → `unar` in the cell below).

In [ ]:
!mkdir -p datasets/mmdocir
!wget -c -q --show-progress -O datasets/mmdocir/doc_pdfs.rar \
    "https://huggingface.co/datasets/MMDocIR/MMDocIR_Evaluation_Dataset/resolve/main/doc_miscellaneous/doc_pdfs.rar?download=true"
!unrar x -o+ -idq datasets/mmdocir/doc_pdfs.rar datasets/mmdocir/
!rm -f datasets/mmdocir/doc_pdfs.rar

In [ ]:
from datasets import load_dataset

# We only need the per-document QA annotations (313 docs, one JSON line each).
# pages.parquet (20,395 rows) and layouts.parquet (170,338 rows) are intentionally
# skipped -- they're large and we'll re-derive equivalent info from the PDFs ourselves.
REPO_ID = "MMDocIR/MMDocIR_Evaluation_Dataset"
annotations = load_dataset(REPO_ID, data_files="MMDocIR_annotations.jsonl", split="train")


# Annotations ship as one row *per document* with a nested `questions` list.
# We explode that into one row *per question* so the resulting DataFrame can be
# treated as a flat eval set. Each output row has:
#   doc_name        - source PDF filename (joins to datasets/mmdocir/doc_pdfs/)
#   domain          - high-level corpus tag (e.g. "Academic paper", "Financial report")
#   Q / A           - question text and ground-truth answer string
#   type            - question type label(s) (e.g. ['Chart'], 'text-only')
#   page_id         - list of GT page indices the answer can be found on
#   layout_mapping  - list of {page, page_size, bbox} regions for fine-grained eval
def _flatten_questions(batch):
    out = {k: [] for k in ("doc_name", "domain", "query", "answer", "type", "page_id", "layout_mapping")}
    for doc_name, domain, questions in zip(batch["doc_name"], batch["domain"], batch["questions"]):
        for q in questions:
            out["doc_name"].append(doc_name)
            out["domain"].append(domain)
            out["query"].append(q["Q"])
            out["answer"].append(q["A"])
            out["type"].append(q["type"])
            out["page_id"].append(q["page_id"])
            out["layout_mapping"].append(q["layout_mapping"])
    return out


mmdocir_questions = annotations.map(
    _flatten_questions,
    batched=True,
    remove_columns=annotations.column_names,
).to_pandas()

mmdocir_questions.describe()

In [ ]:
import pandas as pd
import re
import subprocess
from pathlib import Path

import logging
logging.getLogger("pypdf").setLevel(logging.ERROR)

PDF_DIR = "datasets/mmdocir/doc_pdfs"

# A few helpers for slicing the full 1,658-row eval set down to something
# tractable. Indexing every PDF takes a while (each one runs through Unstructured
# `hi_res` + an LLM summary pass), so for a tutorial we prefer a small subset
# rather than the full corpus.
def filter_dataset_by_files(num_files: int, seed: int | None = 10934) -> pd.DataFrame:
    """Randomly pick `num_files` unique `doc_name`s and return all their Q&A rows.

    Args:
        num_files: Number of unique source documents to sample.
        seed: RNG seed for reproducibility. Pass None for non-deterministic sampling.

    Raises:
        ValueError: if `num_files` exceeds the number of unique documents available.
    """
    unique_docs = mmdocir_questions["doc_name"].drop_duplicates()
    if num_files > len(unique_docs):
        raise ValueError(
            f"Requested {num_files} files but only {len(unique_docs)} unique docs are available."
        )
    sampled = unique_docs.sample(n=num_files, random_state=seed)
    # Shuffle the resulting rows so questions from the same doc aren't adjacent.
    return (
        mmdocir_questions[mmdocir_questions["doc_name"].isin(sampled)]
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )


def filter_dataset_by_domains(domains: listtype[str]) -> pd.DataFrame:
    """Return all Q&A rows whose `domain` is in the given list (e.g. only finance docs)."""
    return mmdocir_questions[mmdocir_questions["domain"].isin(domains)].reset_index(drop=True)


def _pdf_page_count(pdf_path: str) -> int:
    """Return the number of pages in a PDF.

    Uses `pypdf` if available (installed via `unstructured[all-docs]`) and falls
    back to the `pdfinfo` CLI (from poppler-utils) so this works regardless of
    which dependency happens to be present.
    """
    try:
        from pypdf import PdfReader
        return len(PdfReader(pdf_path).pages)
    except Exception:
        out = subprocess.check_output(["pdfinfo", pdf_path], text=True, stderr=subprocess.DEVNULL)
        m = re.search(r"^Pages:\s+(\d+)", out, re.M)
        if not m:
            raise RuntimeError(f"Could not determine page count for {pdf_path}")
        return int(m.group(1))


def filter_dataset_by_smallest_files(
    num_files: int,
    pdf_dir: str = PDF_DIR,
    seed: int | None = 10934,
) -> pd.DataFrame:
    """Pick the `num_files` PDFs with the **fewest pages** and return all their Q&A rows.

    Handy when you want the cheapest-to-index slice of the corpus (parsing cost
    scales roughly with page count). Ties are broken by `doc_name` for stable,
    reproducible ordering; output rows are then shuffled with `seed`.

    Args:
        num_files: Number of source documents to pick (smallest first by page count).
        pdf_dir: Directory holding the source PDFs (joined with `doc_name`).
        seed: RNG seed for shuffling the output rows. Pass None for non-deterministic order.

    Raises:
        ValueError: if `num_files` exceeds the number of unique documents available.
    """
    unique_docs = mmdocir_questions["doc_name"].drop_duplicates().tolist()
    if num_files > len(unique_docs):
        raise ValueError(
            f"Requested {num_files} files but only {len(unique_docs)} unique docs are available."
        )

    page_counts: list[tuple[str, int]] = []
    for doc in unique_docs:
        pdf_path = str(Path(pdf_dir) / doc)
        try:
            page_counts.append((doc, _pdf_page_count(pdf_path)))
        except Exception as e:
            # Don't let one bad PDF kill the selection; just send it to the back.
            print(f"  ! could not read {doc}: {e}; skipping")
            page_counts.append((doc, 10**9))

    page_counts.sort(key=lambda x: (x[1], x[0]))
    sampled = [doc for doc, _ in page_counts[:num_files]]
    print(
        "Smallest selected PDFs (pages): "
        + ", ".join(f"{d}={n}" for d, n in page_counts[:num_files])
    )

    return (
        mmdocir_questions[mmdocir_questions["doc_name"].isin(sampled)]
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

# Evaluate on a small sample so the pipeline runs end-to-end quickly.
# `filter_dataset_by_smallest_files` keeps wall-clock low by picking the
# shortest PDFs first; swap to `filter_dataset_by_files` for a random sample
# or `filter_dataset_by_domains` to slice by corpus tag.
eval_dataset = filter_dataset_by_smallest_files(num_files=15)
print(
    f"eval_dataset: {len(eval_dataset):4d} questions across "
    f"{eval_dataset['doc_name'].nunique()} document(s)"
)
eval_dataset.head()

## 2. Create the RapidFire experiment

In [ ]:
experiment = Experiment(experiment_name="exp1-multimodal-doc-ir", mode="evals")

## 3. Build the RAG pipeline, piece by piece

The next cells assemble an **`RFLangChainRagSpec`** — RapidFire's declarative wrapper around a LangChain RAG pipeline:

```
PDFs ──► document_loader ──► multimodal_processor ──► text_splitter
                                                          │
                            artifact_storage_cfg ◄────────┤  (text/tables/images)
                                    │                     ▼
                                    │              vector_store_cfg
                                    │                     │
                                    │                     ▼
                                    │           search_cfg + reranker_cfg
                                    │                     │
                                    │                     ▼
                                    └──────► Retrieve Original Artifacts
                                                          │
                                                          ▼
                                                     VLM Generator
```

Any value below can be wrapped in `List(...)` or `RFGridSearch(...)` to sweep configurations in parallel — that's RapidFire's hyperparallelism in action.

### 3a. Document loader — parsing PDFs into typed elements

[`UnstructuredPDFLoader`](https://docs.unstructured.io/open-source/core-functionality/partitioning) (wrapped in `DirectoryLoader`) with two key knobs:

- **`mode="elements"`** — one `Document` per layout element (paragraph, table, image, ...), each tagged with `metadata["category"]` so downstream stages can route to the right summarizer.
- **`strategy="hi_res"`** — layout detection (Detectron2 / YOLOX) + OCR. Slower than `"fast"`, but the only strategy that reliably recovers tables and figures.

`glob` restricts parsing to the PDFs in `eval_dataset` — no full-directory scan.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, UnstructuredPDFLoader
from unstructured.cleaners.core import (
    clean_bullets,
    clean_extra_whitespace,
    clean_dashes,
    clean_trailing_punctuation,
)

# Glob accepts a list of literal filenames; only these PDFs will be parsed.
source_files = eval_dataset["doc_name"].unique().tolist()

document_loader = DirectoryLoader(
    path="datasets/mmdocir/doc_pdfs/",
    glob=source_files,
    loader_cls=UnstructuredPDFLoader,
    loader_kwargs={
        # --- Element-level parsing ---------------------------------------
        "mode": "elements",                       # one Document per layout element
        "strategy": "hi_res",                     # run layout detection + OCR
        "include_page_breaks": True,              # emit PageBreak elements (helpful for citation)
        "post_processors": [                      # cheap text cleaners run on every element
            clean_bullets,
            clean_extra_whitespace,
            clean_dashes,
            clean_trailing_punctuation,
        ],

        # --- Layout detection Model --------------------------------------------
        "hi_res_model_name": "yolox",             # or "detectron2_onnx"

        # --- Tables + figures --------------------------------------------
        "infer_table_structure": True,            # produce HTML for tables (used for table summaries)
        "extract_image_block_types": ["Image", "Table"],
        "extract_image_block_to_payload": True,   # inline base64 payload -- needed by the vision LLM

        # --- Chunking (Unstructured-side, before LangChain's text splitter) -
        "chunking_strategy": "by_title",          # group elements under their nearest title
        "max_characters": 10000,                  # hard cap per chunk
        "combine_text_under_n_chars": 2000,       # merge tiny adjacent chunks
        "new_after_n_chars": 6000,                # soft target chunk size
    },
    sample_seed=1137,
)

### 3b. Multimodal summarizer — collapse every modality to text

Text embedders only speak text, but our docs have prose, tables, and images. Classic multi-vector RAG: summarize each non-text element with an LLM, embed the summary, keep the original addressable at query time.

- **Text** → light summary (still embedded as text).
- **Tables** → LLM description (embedded); original HTML kept in storage.
- **Images** → vision-LLM caption (embedded); base64 image kept in storage.

`RFAPIModelConfig` wraps remote LLM endpoints (OpenAI, Anthropic, Google, ...) with auth, retries, and per-key rate limits — all three summarizers share one client. Use `RFvLLMModelConfig` for a local model instead.

In [ ]:
# Prompts intentionally tell the model to skip preambles like "Here is a summary..."
# because we embed the raw output verbatim -- preambles are noise in vector space.
text_or_table_instructions = """
You are an assistant tasked with summarizing tables and text.
Give a concise summary of the table or text.

Respond only with the summary, no additional comment.
Do not start your message by saying "Here is a summary" or anything like that.
Just give the summary as it is.

Table or text chunk:
"""

image_instructions = """
You are an assistant tasked with summarizing images.
Describe the image in detail.

Respond only with the summary, no additional comments.
Do not start your message by saying "Here is a summary" or anything like that.

Image:
"""

# One shared remote-LLM client used by all three summarizers. Pinning the
# rate-limit knobs here keeps RapidFire's scheduler from over-saturating the
# upstream API when a sweep fans out many concurrent runs.
summarizer_config = RFAPIModelConfig(
    client_config={"max_retries": 2},
    endpoint_config={
        "provider": "openai",
        "api_key_name": "my_openai_key",
        "api_key": OPENAI_API_KEY,
        "endpoint": {
            "model": "gpt-5.4-nano",                 # vision-capable, cheap
            "name": "summarizer-gpt-5.4-nano",
            "usage_tracking": True,                  # log token usage to MLflow
        },
    },
    model_config={"max_completion_tokens": 2048},
    rpm_limit=30_000,                                 # requests-per-minute ceiling
    tpm_limit=180_000_000,                            # tokens-per-minute ceiling
    rag=None,                                         # not used: this client only summarizes
    prompt_manager=None,                              # not used: instructions are inlined above
)

# RFLangChainRagSpec routes each Unstructured element category to one of these
# three summarizers based on its detected modality.
multimodal_processor = {
    "text_summarizer_cfg":  {"instructions": text_or_table_instructions, "generator": summarizer_config},
    "image_summarizer_cfg": {"instructions": image_instructions,         "generator": summarizer_config},
    "table_summarizer_cfg": {"instructions": text_or_table_instructions, "generator": summarizer_config},
}

### 3c. Artifact storage — keep heavy originals out of the index

Image base64 payloads, table HTML, and raw text bodies can each be tens or hundreds of KB; inlining them in vector-store metadata bloats the index and breaks per-record size limits.

RapidFire offloads each artifact to cloud object storage (S3 or GCS) at indexing time and stores only the returned URI in metadata. At query time the URI resolves back so the LLM sees the original artifact, not just its summary. Pass `artifact_storage_cfg=None` to keep originals inline.

**Configure the bucket**

- **GCS** (this notebook): `pip install google-cloud-storage` + `gcloud auth application-default login`.
- **S3**: `pip install boto3` + standard AWS creds. Swap `backend: "gcs"` → `"s3"`.

In [ ]:
# Artifacts land at gs://<bucket>/<prefix>/<text|image|table>/<rf_doc_id>/document.
# Auth uses the standard GCS credential chain (gcloud ADC, GOOGLE_APPLICATION_CREDENTIALS,
# attached service account).
artifact_storage_cfg = {
    "backend": "gcs",
    "bucket": "rapidfire-sandbox-us-west1",
    "prefix": "mm-rag/artifacts",
}

### 3d. Text splitter — token-aware chunking of the summaries

Split each summary into ~512-token chunks with 32-token overlap. Counting in tokens (via the GPT-2 tokenizer) — not characters — bounds chunks against the embedder's input window and avoids the diluted-embedding problem you get from very long inputs.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# `from_tiktoken_encoder` makes chunk_size / chunk_overlap count tokens (not chars).
# 512-token chunks fit comfortably inside a 8K-context embedding model, and the
# 32-token overlap preserves enough context across boundaries for chunk recall.
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="gpt2",
    chunk_size=512,
    chunk_overlap=32,
)

### 3e. Vector store — embed and index in Pinecone

Embed with OpenAI's `text-embedding-3-small` (1536-dim) and upsert to a **Pinecone serverless** index — no DB process to manage, scales to zero between runs, and LangChain handles index creation + batched upserts. Swap `type` to `chroma`, `weaviate`, or `pgvector` to change backends; the rest of the pipeline doesn't care.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from pinecone import ServerlessSpec

vector_store_cfg = {
    # Embedding model: text-embedding-3-small is 1536-dim and ~5x cheaper than
    # -3-large with only modestly worse retrieval on most benchmarks.
    "embedding_cfg": {
        "class": OpenAIEmbeddings,
        "model": "text-embedding-3-small",
        "api_key": OPENAI_API_KEY
    },

    # Pinecone serverless index. The index is created on first upsert if missing.
    "type": "pinecone",
    "pinecone_api_key": PINECONE_API_KEY,    # or set the PINECONE_API_KEY env var
    "spec": ServerlessSpec(cloud="gcp", region="us-central1"),
    "metric": "cosine",                       # "cosine" matches OpenAI's normalized embeddings
    "batch_size": 1024,                       # upsert chunks in batches of 1024
}

### 3f. Retrieval + reranking — quality without latency

Two-stage retrieval, the standard recipe for serious RAG quality:

1. **Vector search** — top-*k* candidates from Pinecone. Fast but approximate.
2. **Cross-encoder rerank** — rescore the *k* with `cross-encoder/ms-marco-MiniLM-L6-v2`, which sees `(query, candidate)` jointly. Much better than a bi-encoder for relevance, too slow to run over the full index but fine on a small candidate pool.

In [ ]:
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

# Stage 1: top-10 candidates from Pinecone. Sweep two retrieval strategies side-by-side:
# similarity (pure dense) and MMR (diversity-aware re-ranking of the candidate pool).
search_cfg = List([{"type": "similarity", "k": 10}, {"type": "mmr", "k": 10}])

# Stage 2: rerank the top-10 with a cross-encoder and keep the top-3 for the LLM.
# Set device="cuda" if you have a GPU; cross-encoders batch well and benefit a lot from it.
reranker_cfg = {
    "class": CrossEncoderReranker,
    "model_name": "cross-encoder/ms-marco-MiniLM-L6-v2",
    "model_kwargs": {"device": "cpu"},
    "top_n": 3,
}

### 3g. Putting it all together — `RFLangChainRagSpec`

One declarative spec bundles the components above into a pipeline RapidFire's controller can build, run, and (with a sweep) parallelize.

In [ ]:
from langchain_core.documents import Document

batch_size = 16

# Single declarative spec describing the entire RAG pipeline.
# RapidFire's controller will build, run, and track this end-to-end.
rag_cpu = RFLangChainRagSpec(
    # document_loader can be a python list of different types of loaders (PDF, JSON, XML, etc.) for simultaneous processing in a single run.
    # python list of laoders can be wrapped in List(...) for sweeping different loaders in parallel across multiple runs.
    document_loader=[document_loader],          # 3a: PDF -> typed elements; 
    multimodal_processor=multimodal_processor,  # 3b: elements -> text summaries
    text_splitter=text_splitter,                # 3d: summaries -> ~512-token chunks
    vector_store_cfg=vector_store_cfg,          # 3e: chunks -> Pinecone index
    search_cfg=search_cfg,                      # 3f: similarity / MMR retrieval, top-k
    reranker_cfg=reranker_cfg,                  # 3f: cross-encoder rerank, top-n
    artifact_storage_cfg=artifact_storage_cfg,  # 3c: offload originals to GCS
)

### Define Data Processing and Postprocessing Functions

In [ ]:
def sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Function to prepare the final inputs given to the generator model"""

    instructions = (
        "You are an assistant answering questions about a wide range of topics. "
        "The user's question is followed by retrieved context that may include "
        "summaries, original text passages, table HTML, and figure images. "
        "Use the context to answer concisely. Cite which artifact "
        "(text / table / image) you relied on. If the answer cannot be found "
        "in the context, say so explicitly."
    )

    # Perform batched retrieval over all queries; returns a list of lists of k documents per query
    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)

    # Extract the (source_doc, page_number) pairs from the retrieved context
    retrieved_documents = [
        [
            (
                doc.metadata["source"].split("/")[-1], 
                doc.metadata["page_number"]
            ) 
            for doc in docs
        ] for docs in all_context
    ]

    ground_truth_documents = [
        [
            (
                _doc_name, 
                _page_number
            ) 
            for _page_number in _pages
        ] for _doc_name, _pages in zip(batch["doc_name"], batch["page_id"])
    ]

    # Construct multimodal prompts
    prompts = []
    for _query, _docs in zip(batch["query"], all_context):
        parts = []
        for doc in _docs:
            doc_type = doc.metadata["document_type"] # This is automatically generated by RapidFire
            doc_id = doc.metadata.get("rf_doc_id", "?") # This is automatically generated by RapidFire

            if doc_type == "image":
                # Get the image from the storage client
                b64 = rag.storage_client.get_image_base64(doc.metadata["image_source"])
                artifact_part = {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{b64}"},
                }
            elif doc_type == "table":
                # Get the HTML from the storage client
                html = rag.storage_client.get_html(doc.metadata["table_source"])
                artifact_part = {
                    "type": "text",
                    "text": f"[Table {doc_id[:8]}... HTML]\n{html}",
                }
            else:
                # Get the original text body from the storage client. 
                text = rag.storage_client.get_text(doc.metadata["text_source"])
                artifact_part = {
                    "type": "text",
                    "text": f"[Text {doc_id[:8]}...]\n{text}",
                }

            # Optionally, include the summary of the artifact
            parts.append(
                {
                    "type": "text",
                    "text": f"[Summary of {doc_type} {doc_id[:8]}...]\n{doc.page_content}",
                }
            )
            # Add the artifact part to the prompt
            parts.append(artifact_part)
        
        prompts.append([
            {"role": "system", "content": instructions},
            {"role": "user", "content": [
                {"type": "text", "text": f"Question: {_query}"},
                {"type": "text", "text": f"Retrieved Context:"},
                *parts
            ]}
        ])

    return {
        "prompts": prompts,
        "retrieved_documents": retrieved_documents,
        "ground_truth_documents": ground_truth_documents,
        **batch,
    }

### Define LLM-as-Judge Evaluation Function

In [ ]:
def evaluate_with_llm_judge(
    query_and_context: str,
    answer: str,
    generated_answer: str,
    model: str = "gpt-5.4-mini",
    max_retries: int = 3
) -> Dict[str, float]:
    """Evaluate generated answer using OpenAI as LLM judge

    NOTE: Creates OpenAI client inside function to avoid Ray serialization issues.

    Returns scores for:
    - Relevance: How well the answer addresses the query (0-5)
    - Faithfulness: How well the answer is grounded in the context (0-5)
    - Coherence: How well-structured and clear the answer is (0-5)
    """
    import os
    import json
    import re
    from openai import OpenAI

    # Create client inside function (each Ray actor will have its own)
    openai_client = OpenAI(api_key=OPENAI_API_KEY)

    rubric = """
You are an expert evaluator for question-answering systems. You will evaluate the quality of a generated answer to a question using the retrieved context and the ground truth answer as reference.

Please evaluate the answer on three dimensions, each on a scale of 0-5:

1. RELEVANCE (0-5): How well does the generated answer address the question?
   - 0: Completely irrelevant
   - 1-2: Barely relevant, misses key points
   - 3: Somewhat relevant, addresses some aspects
   - 4: Mostly relevant, addresses most aspects
   - 5: Perfectly relevant, fully addresses the question

2. FAITHFULNESS (0-5): How well is the generated answer grounded in the provided context?
   - 0: Contains fabricated information not in context
   - 1-2: Mostly speculative, minimal use of context
   - 3: Partially grounded in context
   - 4: Mostly grounded in context with minor additions
   - 5: Fully grounded in context, no hallucinations

3. COHERENCE (0-5): How well-structured and clear is the generated answer?
   - 0: Incomprehensible
   - 1-2: Poorly structured, hard to follow
   - 3: Acceptable structure, somewhat clear
   - 4: Well-structured, clear
   - 5: Excellent structure, very clear and professional

Respond ONLY with a JSON object in this exact format:
{{
  "relevance": <score 0-5>,
  "faithfulness": <score 0-5>,
  "coherence": <score 0-5>
}}
"""

    judge_prompt_content = [
        {'type': "text", "text": rubric},
        *query_and_context,
        {'type': "text", "text": f"Ground Truth Answer: {answer}"},
        {'type': "text", "text": f"Generated Answer: {generated_answer}"},
        {'type': "text", "text": "Your evaluation:"},
    ]

    for attempt in range(max_retries):
        try:
            response = openai_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert evaluator. Respond only with valid JSON."},
                    {"role": "user", "content": judge_prompt_content}
                ],
                temperature=0.0,
                max_completion_tokens=8192,
            )

            # Parse the JSON response
            result_text = response.choices[0].message.content.strip()
            # Remove markdown code blocks if present
            result_text = re.sub(r'^```json\s*|\s*```$', '', result_text, flags=re.MULTILINE)
            result = json.loads(result_text)

            # Validate scores are in range
            for key in ['relevance', 'faithfulness', 'coherence']:
                if key not in result or not (0 <= result[key] <= 5):
                    raise ValueError(f"Invalid {key} score: {result.get(key)}")

            return result

        except (json.JSONDecodeError, ValueError, KeyError) as e:
            if attempt == max_retries - 1:
                # Return neutral scores on final failure
                print(f"LLM judge evaluation failed after {max_retries} attempts: {e}")
                return {
                    "relevance": 2.5,
                    "faithfulness": 2.5,
                    "coherence": 2.5
                }
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"Unexpected error in LLM judge: {e}")
                return {
                    "relevance": 2.5,
                    "faithfulness": 2.5,
                    "coherence": 2.5
                }

    # Fallback
    return {
        "relevance": 2.5,
        "faithfulness": 2.5,
        "coherence": 2.5
    }

### Define Custom Eval Metrics Functions

In [ ]:
import math


def compute_ndcg_at_k(retrieved_docs: list, expected_docs: set, k=5):
    """Utility function to compute NDCG@k"""
    relevance = [1 if doc in expected_docs else 0 for doc in retrieved_docs[:k]]
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))

    # IDCG: perfect ranking limited by min(k, len(expected_docs))
    ideal_length = min(k, len(expected_docs))
    ideal_relevance = [1] * ideal_length + [0] * (k - ideal_length)
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_relevance))

    return dcg / idcg if idcg > 0 else 0.0


def compute_rr(retrieved_docs: list, expected_docs: set):
    """Utility function to compute Reciprocal Rank (RR) for a single query"""
    for i, retrieved_doc in enumerate(retrieved_docs):
        if retrieved_doc in expected_docs:
            return 1 / (i + 1)
    return 0


def sample_compute_metrics_fn(batch: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    """Compute eval metrics on retrievals and (when available) generations.

    Retrieval metrics are page-level: each retrieved / ground-truth document is
    a (doc_name, page_number) tuple emitted by `sample_preprocess_fn`, so a hit
    means we surfaced the correct page of the correct PDF.
    """

    precisions, recalls, f1_scores, ndcgs, rrs = [], [], [], [], []
    relevance_scores, faithfulness_scores, coherence_scores = [], [], []
    total_queries = len(batch["query"])

    # Page-level retrieval metrics
    for pred, gt in zip(batch["retrieved_documents"], batch["ground_truth_documents"]):
        expected_set = set(gt)
        retrieved_set = set(pred)

        tp = len(expected_set.intersection(retrieved_set))
        precision = tp / len(retrieved_set) if len(retrieved_set) > 0 else 0
        recall = tp / len(expected_set) if len(expected_set) > 0 else 0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0
        )

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        ndcgs.append(compute_ndcg_at_k(pred, expected_set, k=5))
        rrs.append(compute_rr(pred, expected_set))

    # LLM-as-judge generation metrics (only when generated text is present)
    if "generated_text" in batch:
        for prompt, answer, generated in zip(
            batch["prompts"],
            batch["answer"],
            batch["generated_text"]
        ):
            query_and_context = prompt[1]["content"]  # includes query and retrieved context
            judge_result = evaluate_with_llm_judge(query_and_context, answer, generated)
            relevance_scores.append(judge_result["relevance"])
            faithfulness_scores.append(judge_result["faithfulness"])
            coherence_scores.append(judge_result["coherence"])

    metrics = {
        "Total": {"value": total_queries},
        # Page-level retrieval metrics (scored on (doc_name, page_number) tuples)
        "Page-Level Precision": {"value": sum(precisions) / total_queries},
        "Page-Level Recall": {"value": sum(recalls) / total_queries},
        "Page-Level F1": {"value": sum(f1_scores) / total_queries},
        "Page-Level NDCG@5": {"value": sum(ndcgs) / total_queries},
        "Page-Level MRR": {"value": sum(rrs) / total_queries},
    }

    # LLM-judge generation metrics (answer quality, not retrieval)
    if relevance_scores:
        metrics.update({
            "LLM Relevance": {"value": sum(relevance_scores) / total_queries / 5.0},
            "LLM Faithfulness": {"value": sum(faithfulness_scores) / total_queries / 5.0},
            "LLM Coherence": {"value": sum(coherence_scores) / total_queries / 5.0},
        })

    return metrics

def sample_accumulate_metrics_fn(
    aggregated_metrics: Dict[str, listtype],
) -> Dict[str, Dict[str, Any]]:
    """Accumulate eval metrics across all batches."""

    num_queries_per_batch = [m["value"] for m in aggregated_metrics["Total"]]
    total_queries = sum(num_queries_per_batch)
    algebraic_metrics = [
        "Page-Level Precision", "Page-Level Recall", "Page-Level F1",
        "Page-Level NDCG@5", "Page-Level MRR",
        "LLM Relevance", "LLM Faithfulness", "LLM Coherence",
    ]

    return {
        "Total": {"value": total_queries},
        **{
            metric: {
                "value": sum(
                    m["value"] * queries
                    for m, queries in zip(
                        aggregated_metrics[metric], num_queries_per_batch
                    )
                )
                / total_queries,
                "is_algebraic": True,
                "value_range": (0, 1),
            }
            for metric in algebraic_metrics
        },
    }

### Define generator configs to sweep

In [ ]:
# Three generator configs swept in parallel. OpenAI routes through MLflow Gateway;
# Gemini and Anthropic use provider="openai" + api_base_url to hit each provider's
# OpenAI-compatible endpoint directly (the gateway doesn't translate multimodal
# content[] for non-OpenAI providers).
openai_config = RFAPIModelConfig(
    client_config={"max_retries": 2},
    endpoint_config={
        "provider": "openai",
        "api_key_name": "my_openai_key",
        "api_key": OPENAI_API_KEY,
        "endpoint": {
            "model": "gpt-5.4-nano",
            "name": "gpt-5.4-nano",
            "usage_tracking": True,
        },
    },
    model_config={
        "max_completion_tokens": 8192,
    },
    rpm_limit=30_000,
    tpm_limit=180_000_000,
    rag=rag_cpu,
    prompt_manager=None,
)

gemini_config = RFAPIModelConfig(
    client_config={"max_retries": 2},
    endpoint_config={
        "provider": "openai", # ← passthrough; not "gemini"
        "api_key_name": "my_gemini_key3",
        "api_key": GOOGLE_API_KEY,
        "api_base_url": "https://generativelanguage.googleapis.com/v1beta/openai/", # ← openai-compatible endpoint
        "endpoint": {
            "model": "gemini-2.5-flash-lite",
            "name": "openai-gemini-mm",
            "usage_tracking": True,
        },
    },
    model_config={
        "max_completion_tokens": 8192,
    },
    rpm_limit=30_000,
    tpm_limit=30_000_000,
    rag=rag_cpu,
    prompt_manager=None,
)

anthropic_config = RFAPIModelConfig(
    client_config={"max_retries": 2},
    endpoint_config={
        "provider": "openai",  # ← passthrough; not "anthropic"
        "api_key_name": "my_anthropic_key",
        "api_key": ANTHROPIC_API_KEY,
        "api_base_url": "https://api.anthropic.com/v1/", # ← openai-compatible endpoint
        "endpoint": {
            "model": "claude-sonnet-4-6",
            "name": "openai-anthropic-mm",
            "usage_tracking": True,
        },
    },
    model_config={"max_completion_tokens": 8192},
    rpm_limit=2000,
    tpm_limit=960_000,   
    rag=rag_cpu,
    prompt_manager=None,
)

config_set = {
    "api_config": List([openai_config, gemini_config, anthropic_config]),
    "batch_size": batch_size,
    "preprocess_fn": sample_preprocess_fn,
    "postprocess_fn": None,
    "compute_metrics_fn": sample_compute_metrics_fn,
    "accumulate_metrics_fn": sample_accumulate_metrics_fn,
    "online_strategy_kwargs": {
        "strategy_name": "normal",
        "confidence_level": 0.95,
        "use_fpc": True,
    },
}

### Create Config Group

In [ ]:
# Grid search across all multi-value knobs (api_config x search_cfg).
config_group = RFGridSearch(config_set)

### Run Multi-Config Evals

In [ ]:
# Run every config in the group; dataset is sharded into 4 chunks for swap-based scheduling.
results = experiment.run_evals(
    config_group=config_group,
    dataset=Dataset.from_pandas(eval_dataset),
    num_shards=4,
    seed=42,
)

### View Results

In [ ]:
# Convert results dict to DataFrame
results_df = pd.DataFrame([
    {k: v['value'] if isinstance(v, dict) and 'value' in v else v for k, v in {**metrics_dict, 'run_id': run_id}.items()}
    for run_id, (_, metrics_dict) in results.items()
])

results_df

### End Experiment

In [ ]:
experiment.end()

### View RapidFire AI Log Files

In [ ]:
# Get the experiment-specific log file
log_file = experiment.get_log_file_path()

print(f"📄 Log File: {log_file}")
print()

if log_file.exists():
    print("=" * 80)
    print(f"Last 30 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())
else:
    print(f"❌ Log file not found: {log_file}")